In [ ]:
!pip install --upgrade torch transformers evaluate

In [ ]:
!pip install datasets pandas huggingface_hub

In [ ]:
import pandas as pd
from datasets import Dataset, Features, ClassLabel, Value
from huggingface_hub import HfApi, HfFolder
from sklearn.model_selection import train_test_split

In [ ]:
#hf_token = xxx

In [ ]:
!huggingface-cli login

### Loading Dataset

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from datasets import load_dataset, DatasetDict
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np
from tqdm.auto import tqdm
import time
from huggingface_hub import HfApi, login

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load dataset from Hugging Face Hub
# load_dataset fetches data and caches it locally for reuse
reporting = load_dataset("JayShah07/reporting_final_dataset")

print("Dataset structure:")
print(reporting)
print("\nFirst training example:")
print(reporting["train"][0])

print("\nDataset splits after creating test set:")
print(f"Train: {len(reporting['train'])} samples")
print(f"Validation: {len(reporting['validation'])} samples")
print(f"Test: {len(reporting['test'])} samples")


In [ ]:
module_labels = [
    'holdings',
    'capital_gains',
    'scheme_wise_returns',
    'investment_account_wise_returns',
    'portfolio_update',
    'None_module'
]

date_labels = [
    'Current Year',
    'Previous Year',
    'Daily',
    'Monthly',
    'Weekly',
    'Yearly',
    'None_date'
]

# Create label to index mappings for efficient lookup
# These dictionaries convert label names to integer indices
module_label2id = {label: idx for idx, label in enumerate(module_labels)}
date_label2id = {label: idx for idx, label in enumerate(date_labels)}

# Create index to label mappings for decoding predictions
# These dictionaries convert integer indices back to label names
module_id2label = {idx: label for idx, label in enumerate(module_labels)}
date_id2label = {idx: label for idx, label in enumerate(date_labels)}

print(f"\nModule labels ({len(module_labels)}): {module_labels}")
print(f"Date labels ({len(date_labels)}): {date_labels}")

print(f"\nModule label2id: {module_label2id}")
print(f"Module id2label: {module_id2label}")

print(f"\nDate label2id: {date_label2id}")
print(f"Date id2label: {date_id2label}")

In [ ]:
# Load TinyBERT tokenizer
# AutoTokenizer.from_pretrained: loads pre-trained tokenizer config
# This tokenizer converts text to token IDs that the model understands
tokenizer = AutoTokenizer.from_pretrained("huawei-noah/TinyBERT_General_4L_312D")

print(f"\nTokenizer vocabulary size: {tokenizer.vocab_size}")
print(f"Max length: {tokenizer.model_max_length}")

from transformers import AutoModel

model = AutoModel.from_pretrained("huawei-noah/TinyBERT_General_4L_312D")
print(model.config.max_position_embeddings)

In [ ]:
def tokenize_function(examples):
    """
    Tokenizes input text and prepares labels for both classification heads.

    Parameters explained:
    - examples: batch of data samples from dataset
    - tokenizer: converts text to token IDs
    - padding='max_length': pads all sequences to same length
    - truncation=True: cuts text longer than max_length
    - max_length=128: maximum sequence length (balances speed vs context)
    - return_tensors=None: returns Python lists (converted to tensors later)

    Returns:
    - Dictionary with input_ids, attention_mask, module_labels, date_labels
    """

    # Tokenize the user queries
    # This converts text like "Show my holdings" to [101, 2265, 2026, 9583, 102]
    tokenized = tokenizer(
        examples["query"],
        padding='max_length',      # Pad shorter sequences to max_length
        truncation=True,            # Truncate longer sequences to max_length
        max_length=256,             # Set maximum sequence length
        return_tensors=None         # Return as Python lists
    )

    # Prepare module labels - find which module is marked as 1
    # For each example, we create a list of module label values
    module_labels_batch = []
    for i in range(len(examples["query"])):
        # Extract values for all module labels for this example
        labels = [examples[label][i] for label in module_labels]
        module_labels_batch.append(labels)

    # Prepare date labels - find which date is marked as 1
    date_labels_batch = []
    for i in range(len(examples["query"])):
        # Extract values for all date labels for this example
        labels = [examples[label][i] for label in date_labels]
        date_labels_batch.append(labels)

    # Add labels to tokenized output
    tokenized["module_labels"] = module_labels_batch
    tokenized["date_labels"] = date_labels_batch

    return tokenized

# Apply tokenization to all dataset splits
# batched=True: processes multiple examples at once (faster)
# batch_size=32: number of examples to process together
tokenized_datasets = reporting.map(
    tokenize_function,
    batched=True,
    batch_size=32,
    desc="Tokenizing datasets"
)

# Set format to PyTorch tensors for training
# This converts lists to torch.Tensor objects
# columns: specifies which fields to include
tokenized_datasets.set_format(
    type='torch',
    columns=['input_ids', 'attention_mask', 'module_labels', 'date_labels']
)

print("\nTokenized dataset format:")
print(tokenized_datasets["train"][0])


In [ ]:
# ============================================================================
# SECTION 9: DEFINE CUSTOM MODEL WITH TWO CLASSIFICATION HEADS
# ============================================================================

class TinyBERTDualClassifier(nn.Module):
    """
    Custom model with TinyBERT encoder and two classification heads.

    Architecture:
    1. TinyBERT encoder (4 layers, 312 hidden dims) - shared feature extractor
    2. Module classifier head - predicts module category (6 classes)
    3. Date classifier head - predicts date category (7 classes)

    nn.Module: base class for all neural network modules in PyTorch
    """

    def __init__(self, num_module_labels, num_date_labels, dropout_rate=0.1):
        """
        Initialize the model.

        Parameters:
        - num_module_labels: number of module classes (5)
        - num_date_labels: number of date classes (7)
        - dropout_rate: probability of dropping neurons (prevents overfitting)
        """
        # Call parent class constructor
        super(TinyBERTDualClassifier, self).__init__()

        # Load pre-trained TinyBERT encoder
        # This contains the transformer layers that understand language
        self.encoder = AutoModel.from_pretrained("huawei-noah/TinyBERT_General_4L_312D")

        # Get hidden size from encoder config (312 for TinyBERT)
        # hidden_size: dimension of the encoder's output vectors
        self.hidden_size = self.encoder.config.hidden_size

        # Dropout layer for regularization
        # nn.Dropout: randomly sets input elements to 0 during training
        # p=dropout_rate: probability of dropping each element
        self.dropout = nn.Dropout(p=dropout_rate)

        # Module classification head
        # nn.Linear: fully connected layer (y = xW^T + b)
        # in_features: input dimension (312)
        # out_features: output dimension (6 module classes)
        self.module_classifier = nn.Linear(
            in_features=self.hidden_size,
            out_features=num_module_labels
        )

        # Date classification head
        # Separate classifier for date categories (7 classes)
        self.date_classifier = nn.Linear(
            in_features=self.hidden_size,
            out_features=num_date_labels
        )

    def forward(self, input_ids, attention_mask):
        """
        Forward pass through the model.

        Parameters:
        - input_ids: tokenized input sequences [batch_size, seq_length]
        - attention_mask: mask for padding tokens [batch_size, seq_length]

        Returns:
        - module_logits: raw scores for module classes [batch_size, 6]
        - date_logits: raw scores for date classes [batch_size, 7]

        Flow:
        1. Text → Encoder → Contextualized representations
        2. Take [CLS] token representation (first token)
        3. Apply dropout
        4. Pass through both classifiers
        """

        # Pass inputs through TinyBERT encoder
        # output_hidden_states=True: returns all layer outputs (not used here)
        # outputs.last_hidden_state: final layer output [batch_size, seq_length, 312]
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Extract [CLS] token representation (first token of sequence)
        # [CLS] token captures the overall meaning of the sentence
        # Shape: [batch_size, 312]
        cls_output = outputs.last_hidden_state[:, 0, :]

        # Apply dropout for regularization
        # During training, randomly drops features
        # During evaluation, scales features appropriately
        cls_output = self.dropout(cls_output)

        # Get logits (raw scores) from both classification heads
        # Logits are unnormalized scores (before softmax)
        module_logits = self.module_classifier(cls_output)  # [batch_size, 6]
        date_logits = self.date_classifier(cls_output)      # [batch_size, 7]

        return module_logits, date_logits


In [ ]:
# ============================================================================
# SECTION 10: INITIALIZE MODEL
# ============================================================================
print(len(module_labels))
print(len(date_labels))
# Create model instance with correct number of labels
model = TinyBERTDualClassifier(
    num_module_labels=len(module_labels),  # 6 module classes
    num_date_labels=len(date_labels),      # 7 date classes
    dropout_rate=0.2                        # 10% dropout probability
)

# Move model to GPU/CPU
# .to(device): transfers all model parameters to specified device
model = model.to(device)

# Count total parameters
# sum(...): adds up all parameter counts
# p.numel(): returns number of elements in parameter tensor
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel initialized:")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size: ~{total_params * 4 / 1024 / 1024:.2f} MB (FP32)")


In [ ]:
# Batch size for training and evaluation
# Smaller batch sizes use less memory but may train slower
BATCH_SIZE = 16

# Create DataLoaders for each split
# DataLoader: provides batching, shuffling, and parallel loading
# shuffle=True: randomizes order each epoch (prevents overfitting)
# shuffle=False: keeps order consistent (for reproducible evaluation)
train_dataloader = DataLoader(
    tokenized_datasets["train"],
    batch_size=BATCH_SIZE,
    shuffle=True  # Shuffle training data each epoch
)

val_dataloader = DataLoader(
    tokenized_datasets["validation"],
    batch_size=BATCH_SIZE,
    shuffle=False  # Don't shuffle validation data
)

test_dataloader = DataLoader(
    tokenized_datasets["test"],
    batch_size=BATCH_SIZE,
    shuffle=False  # Don't shuffle test data
)

print(f"\nDataLoader batch sizes:")
print(f"Training batches: {len(train_dataloader)}")
print(f"Validation batches: {len(val_dataloader)}")
print(f"Test batches: {len(test_dataloader)}")


In [ ]:
# ============================================================================
# SECTION 12: SETUP TRAINING COMPONENTS
# ============================================================================

# Training hyperparameters
# These control how the model learns
EPOCHS = 10                    # Number of complete passes through training data
LEARNING_RATE = 2e-5          # Step size for parameter updates (small for fine-tuning)
WEIGHT_DECAY = 0.01           # L2 regularization strength (prevents large weights)

# Loss function for multi-class classification
# nn.CrossEntropyLoss: combines softmax + negative log likelihood
# Expects raw logits (not probabilities) and class indices as targets
criterion = nn.CrossEntropyLoss()

# Optimizer - updates model parameters based on gradients
# AdamW: Adam optimizer with weight decay (better for transformers)
# lr: learning rate (how much to update parameters)
# weight_decay: coefficient for L2 penalty on weights
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Calculate total training steps for learning rate scheduling
# num_training_steps: total number of optimizer updates
num_training_steps = len(train_dataloader) * EPOCHS

# Learning rate scheduler - gradually decreases learning rate
# Warmup: linearly increases LR for first 10% of training
# Then: linearly decreases LR to 0 by end of training
# This helps model converge better
num_warmup_steps = int(0.1 * num_training_steps)  # 10% warmup
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

print(f"\nTraining configuration:")
print(f"Epochs: {EPOCHS}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Weight decay: {WEIGHT_DECAY}")
print(f"Total training steps: {num_training_steps}")
print(f"Warmup steps: {num_warmup_steps}")

In [ ]:
def evaluate_model(model, dataloader, criterion, device):
    """
    Evaluate model on a dataset.

    Parameters:
    - model: the neural network to evaluate
    - dataloader: DataLoader providing evaluation batches
    - criterion: loss function
    - device: CPU or GPU

    Returns:
    - Dictionary with metrics: loss, accuracy, F1 score for both heads

    Process:
    1. Set model to evaluation mode
    2. Disable gradient computation (saves memory)
    3. Process each batch and collect predictions
    4. Calculate metrics
    """

    # Set model to evaluation mode
    # This disables dropout and batch normalization training behavior
    model.eval()

    # Initialize accumulators
    total_loss = 0
    module_preds_list = []
    module_labels_list = []
    date_preds_list = []
    date_labels_list = []

    # Disable gradient computation for evaluation
    # torch.no_grad(): reduces memory usage and speeds up computation
    with torch.no_grad():
        # Iterate through batches
        for batch in dataloader:
            # Move batch to device (GPU/CPU)
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            module_labels = batch['module_labels'].to(device)
            date_labels = batch['date_labels'].to(device)

            # Forward pass
            module_logits, date_logits = model(input_ids, attention_mask)

            # Convert multi-hot labels to class indices
            # torch.argmax: finds index of maximum value (the true class)
            # dim=1: operate along class dimension
            module_targets = torch.argmax(module_labels, dim=1)
            date_targets = torch.argmax(date_labels, dim=1)

            # Calculate losses for both heads
            module_loss = criterion(module_logits, module_targets)
            date_loss = criterion(date_logits, date_targets)

            # Combined loss (average of both heads)
            loss = (module_loss + date_loss) / 2
            total_loss += loss.item()

            # Get predictions (class with highest score)
            module_preds = torch.argmax(module_logits, dim=1)
            date_preds = torch.argmax(date_logits, dim=1)

            # Collect predictions and labels
            # .cpu(): move from GPU to CPU
            # .numpy(): convert PyTorch tensor to NumPy array
            module_preds_list.extend(module_preds.cpu().numpy())
            module_labels_list.extend(module_targets.cpu().numpy())
            date_preds_list.extend(date_preds.cpu().numpy())
            date_labels_list.extend(date_targets.cpu().numpy())

    # Calculate average loss
    avg_loss = total_loss / len(dataloader)

    # Calculate accuracy (percentage of correct predictions)
    # accuracy_score: compares predictions to true labels
    module_acc = accuracy_score(module_labels_list, module_preds_list)
    date_acc = accuracy_score(date_labels_list, date_preds_list)

    # Calculate F1 score (harmonic mean of precision and recall)
    # average='macro': compute F1 for each class and average
    module_f1 = f1_score(module_labels_list, module_preds_list, average='macro')
    date_f1 = f1_score(date_labels_list, date_preds_list, average='macro')

    return {
        'loss': avg_loss,
        'module_accuracy': module_acc,
        'module_f1': module_f1,
        'date_accuracy': date_acc,
        'date_f1': date_f1,
        'module_preds': module_preds_list,
        'module_labels': module_labels_list,
        'date_preds': date_preds_list,
        'date_labels': date_labels_list
    }

In [ ]:
# ============================================================================
# SECTION 14: TRAINING LOOP
# ============================================================================

print("\n" + "="*80)
print("STARTING TRAINING")
print("="*80)

# Track best validation performance for model saving
best_val_loss = float('inf')  # Initialize to infinity
training_history = []

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    print("-" * 80)

    # =========================
    # TRAINING PHASE
    # =========================

    # Set model to training mode
    # Enables dropout and batch normalization training behavior
    model.train()

    # Initialize epoch metrics
    total_train_loss = 0

    # Progress bar for training
    # tqdm: creates progress bar
    # total: number of iterations
    progress_bar = tqdm(train_dataloader, desc="Training")

    for batch in progress_bar:
        # Move batch to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        module_labels = batch['module_labels'].to(device)
        date_labels = batch['date_labels'].to(device)

        # Zero out gradients from previous step
        # optimizer.zero_grad(): resets gradients to zero
        # Required before each backward pass
        optimizer.zero_grad()

        # Forward pass
        module_logits, date_logits = model(input_ids, attention_mask)

        # Convert multi-hot labels to class indices
        module_targets = torch.argmax(module_labels, dim=1)
        date_targets = torch.argmax(date_labels, dim=1)

        # Calculate losses
        module_loss = criterion(module_logits, module_targets)
        date_loss = criterion(date_logits, date_targets)

        # Combined loss (average of both heads)
        loss = (module_loss + date_loss) / 2

        # Backward pass - compute gradients
        # loss.backward(): computes gradient of loss w.r.t. all parameters
        loss.backward()

        # Gradient clipping to prevent exploding gradients
        # torch.nn.utils.clip_grad_norm_: scales gradients if norm exceeds threshold
        # max_norm=1.0: maximum allowed gradient norm
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # Update parameters
        # optimizer.step(): applies gradients to update parameters
        optimizer.step()

        # Update learning rate
        # scheduler.step(): adjusts learning rate according to schedule
        scheduler.step()

        # Accumulate loss
        total_train_loss += loss.item()

        # Update progress bar
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})

    # Calculate average training loss for epoch
    avg_train_loss = total_train_loss / len(train_dataloader)

    # =========================
    # VALIDATION PHASE
    # =========================

    print("\nEvaluating on validation set...")
    val_metrics = evaluate_model(model, val_dataloader, criterion, device)

    # Print epoch results
    print(f"\nEpoch {epoch + 1} Results:")
    print(f"Training Loss: {avg_train_loss:.4f}")
    print(f"Validation Loss: {val_metrics['loss']:.4f}")
    print(f"\nModule Classification:")
    print(f"  Accuracy: {val_metrics['module_accuracy']:.4f}")
    print(f"  F1 Score: {val_metrics['module_f1']:.4f}")
    print(f"\nDate Classification:")
    print(f"  Accuracy: {val_metrics['date_accuracy']:.4f}")
    print(f"  F1 Score: {val_metrics['date_f1']:.4f}")

    # Save training history
    training_history.append({
        'epoch': epoch + 1,
        'train_loss': avg_train_loss,
        'val_loss': val_metrics['loss'],
        'module_acc': val_metrics['module_accuracy'],
        'module_f1': val_metrics['module_f1'],
        'date_acc': val_metrics['date_accuracy'],
        'date_f1': val_metrics['date_f1']
    })

    # Save best model based on validation loss
    if val_metrics['loss'] < best_val_loss:
        best_val_loss = val_metrics['loss']
        # Save model state dictionary
        # state_dict(): returns dictionary of all parameters
        torch.save(model.state_dict(), 'best_model.pt')
        print(f"✓ New best model saved! (Val Loss: {best_val_loss:.4f})")

print("\n" + "="*80)
print("TRAINING COMPLETED")
print("="*80)


In [ ]:
# ============================================================================
# SECTION 15: LOAD BEST MODEL AND EVALUATE ON TEST SET
# ============================================================================

print("\n" + "="*80)
print("FINAL EVALUATION ON TEST SET")
print("="*80)

# Define the CORRECT label lists (must match your model output)
module_labels = [
    'holdings',
    'capital_gains',
    'scheme_wise_returns',
    'investment_account_wise_returns',
    'portfolio_update',
    'None_module'
]

date_labels = [
    'Current Year',
    'Previous Year',
    'Daily',
    'Monthly',
    'Weekly',
    'Yearly',
    'None_date'
]

# Load best model weights
model.load_state_dict(torch.load('best_model.pt'))
print("✓ Best model loaded")

# Evaluate on test set
print("\nEvaluating on test set...")
test_metrics = evaluate_model(model, test_dataloader, criterion, device)

print(f"\nTest Set Results:")
print(f"Loss: {test_metrics['loss']:.4f}")
print(f"\nModule Classification:")
print(f"  Accuracy: {test_metrics['module_accuracy']:.4f}")
print(f"  F1 Score: {test_metrics['module_f1']:.4f}")
print(f"\nDate Classification:")
print(f"  Accuracy: {test_metrics['date_accuracy']:.4f}")
print(f"  F1 Score: {test_metrics['date_f1']:.4f}")

# Detailed classification reports
print("\n" + "="*80)
print("DETAILED CLASSIFICATION REPORTS")
print("="*80)

print("\nModule Classification Report:")
print(classification_report(
    test_metrics['module_labels'],
    test_metrics['module_preds'],
    target_names=module_labels,  # Now has 6 items matching the 6 classes
    zero_division=0
))

print("\nDate Classification Report:")
print(classification_report(
    test_metrics['date_labels'],
    test_metrics['date_preds'],
    target_names=date_labels,  # Now has 7 items matching the 7 classes
    zero_division=0
))

In [ ]:
# ============================================================================
# SECTION 16: INFERENCE FUNCTION WITH LATENCY ANALYSIS
# ============================================================================

def predict_query(text, model, tokenizer, device, num_runs=100):
    """
    Predict module and date for a single query with latency measurement.

    Parameters:
    - text: input query string
    - model: trained model
    - tokenizer: tokenizer for text processing
    - device: CPU or GPU
    - num_runs: number of times to run inference for latency measurement

    Returns:
    - Dictionary with predictions and latency statistics
    """

    # Set model to evaluation mode
    model.eval()

    # Tokenize input
    # return_tensors='pt': return PyTorch tensors
    inputs = tokenizer(
        text,
        padding='max_length',
        truncation=True,
        max_length=256,
        return_tensors='pt'
    )

    # Move inputs to device
    input_ids = inputs['input_ids'].to(device)
    attention_mask = inputs['attention_mask'].to(device)

    # Warm-up run (first run is often slower)
    with torch.no_grad():
        _ = model(input_ids, attention_mask)

    # Measure latency over multiple runs
    latencies = []

    with torch.no_grad():
        for _ in range(num_runs):
            # Record start time
            start_time = time.time()

            # Forward pass
            module_logits, date_logits = model(input_ids, attention_mask)

            # Synchronize CUDA operations (ensures GPU operations complete)
            if device.type == 'cuda':
                torch.cuda.synchronize()

            # Record end time
            end_time = time.time()

            # Calculate latency in milliseconds
            latency_ms = (end_time - start_time) * 1000
            latencies.append(latency_ms)

    # Get final predictions
    with torch.no_grad():
        module_logits, date_logits = model(input_ids, attention_mask)

        # Get class probabilities using softmax
        # torch.softmax: converts logits to probabilities
        # dim=1: apply along class dimension
        module_probs = torch.softmax(module_logits, dim=1)
        date_probs = torch.softmax(date_logits, dim=1)

        # Get predicted classes
        module_pred = torch.argmax(module_probs, dim=1).item()
        date_pred = torch.argmax(date_probs, dim=1).item()

        # Get confidence scores (probability of predicted class)
        module_confidence = module_probs[0, module_pred].item()
        date_confidence = date_probs[0, date_pred].item()

    # Calculate latency statistics
    latencies = np.array(latencies)

    return {
        'query': text,
        'module': module_id2label[module_pred],
        'module_confidence': module_confidence,
        'date': date_id2label[date_pred],
        'date_confidence': date_confidence,
        'latency_mean_ms': np.mean(latencies),
        'latency_std_ms': np.std(latencies),
        'latency_min_ms': np.min(latencies),
        'latency_max_ms': np.max(latencies),
        'latency_p50_ms': np.percentile(latencies, 50),
        'latency_p95_ms': np.percentile(latencies, 95),
        'latency_p99_ms': np.percentile(latencies, 99)
    }


In [ ]:
# ============================================================================
# SECTION 17: TEST INFERENCE WITH SAMPLE QUERIES
# ============================================================================

print("\n" + "="*80)
print("INFERENCE TESTING WITH LATENCY ANALYSIS")
print("="*80)

# Sample queries for testing
sample_queries = [
    "Show me my current holdings",
    "What are my capital gains for this year?",
    "Give me monthly scheme-wise returns",
    "Show investment account returns for the previous year",
    "Provide daily portfolio updates"
]

print(f"\nRunning inference on {len(sample_queries)} sample queries...")
print("(100 runs per query for latency measurement)\n")

for query in sample_queries:
    result = predict_query(query, model, tokenizer, device, num_runs=100)

    print(f"Query: {result['query']}")
    print(f"Module: {result['module']} (confidence: {result['module_confidence']:.4f})")
    print(f"Date: {result['date']} (confidence: {result['date_confidence']:.4f})")
    print(f"Latency (ms):")
    print(f"  Mean: {result['latency_mean_ms']:.2f} ± {result['latency_std_ms']:.2f}")
    print(f"  Min/Max: {result['latency_min_ms']:.2f} / {result['latency_max_ms']:.2f}")
    print(f"  P50/P95/P99: {result['latency_p50_ms']:.2f} / {result['latency_p95_ms']:.2f} / {result['latency_p99_ms']:.2f}")
    print("-" * 80)


In [ ]:
# ============================================================================
# SECTION 18: SAVE MODEL FOR HUGGING FACE
# ============================================================================

print("\n" + "="*80)
print("SAVING MODEL FOR HUGGING FACE HUB")
print("="*80)

# Create directory for saving
import os
save_directory = "./tinybert_dual_classifier"
os.makedirs(save_directory, exist_ok=True)

# Save model encoder and tokenizer
model.encoder.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)

# Save the classifier heads and configuration
torch.save({
    'module_classifier': model.module_classifier.state_dict(),
    'date_classifier': model.date_classifier.state_dict(),
    'module_labels': module_labels,
    'date_labels': date_labels,
    'module_label2id': module_label2id,
    'date_label2id': date_label2id,
    'hidden_size': model.hidden_size
}, os.path.join(save_directory, 'classifier_heads.pt'))

print(f"✓ Model saved to {save_directory}/")
print("\nSaved files:")
print("  - config.json (encoder config)")
print("  - pytorch_model.bin (encoder weights)")
print("  - tokenizer files")
print("  - classifier_heads.pt (classification heads)")

In [ ]:
# ============================================================================
# SECTION 19: UPLOAD TO HUGGING FACE HUB
# ============================================================================

print("\n" + "="*80)
print("UPLOAD TO HUGGING FACE HUB")
print("="*80)

# Login to Hugging Face (you need to provide your token)
print("\nTo upload to Hugging Face Hub:")
print("1. Get your token from: https://huggingface.co/settings/tokens")
print("2. Run the login cell below with your token")
print("3. Then run the upload cell")

# STEP 1: Login to Hugging Face
# Replace 'your_token_here' with your actual Hugging Face token

from huggingface_hub import login

# Uncomment and add your token:
# login(token='your_token_here')

# OR use notebook_login() for interactive login:
from huggingface_hub import notebook_login
notebook_login()

# STEP 2: Upload to Hugging Face Hub
# Replace 'your-username/tinybert-dual-classifier' with your desired repo name

from huggingface_hub import HfApi

repo_name = "JayShah07/tinybert-dual-classifier"

# Push encoder and tokenizer
model.encoder.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)

# Upload classifier heads
api = HfApi()
api.upload_file(
    path_or_fileobj=os.path.join(save_directory, 'classifier_heads.pt'),
    path_in_repo='classifier_heads.pt',
    repo_id=repo_name,
    repo_type='model'
)

print(f"✓ Model uploaded to: https://huggingface.co/{repo_name}")

In [ ]:
# ============================================================================
# SECTION 20: CREATE MODEL CARD (README.md) FOR HUGGING FACE
# ============================================================================

model_card = f"""---
language: en
license: apache-2.0
tags:
- text-classification
- multi-label-classification
- tinybert
- pytorch
datasets:
- JayShah07/multi_label_reporting
metrics:
- accuracy
- f1
widget:
- text: "Show me my current holdings"
- text: "What are my capital gains for this year?"
- text: "Give me monthly scheme-wise returns"
---

# TinyBERT Dual Classifier for Investment Reporting

This model is a fine-tuned TinyBERT with two classification heads for multi-label classification of investment reporting queries.

## Model Description

- **Base Model**: TinyBERT (huawei-noah/TinyBERT_General_4L_312D)
- **Parameters**: ~14-15M
- **Architecture**: Single encoder with two independent classification heads
- **Task**: Multi-label classification (Module + Date)

## Labels

**Module Labels (6 classes)**:
- holdings
- capital_gains
- scheme_wise_returns
- investment_account_wise_returns
- portfolio_update
- None_module

**Date Labels (7 classes)**:
- Current Year
- Previous Year
- Daily
- Monthly
- Weekly
- Yearly
- None_date

## Performance

**Test Set Results**:
- Module Accuracy: {test_metrics['module_accuracy']:.4f}
- Module F1 Score: {test_metrics['module_f1']:.4f}
- Date Accuracy: {test_metrics['date_accuracy']:.4f}
- Date F1 Score: {test_metrics['date_f1']:.4f}

## Usage
```python
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn as nn

# Define model class
class TinyBERTDualClassifier(nn.Module):
    def __init__(self, num_module_labels, num_date_labels, dropout_rate=0.1):
        super(TinyBERTDualClassifier, self).__init__()
        self.encoder = AutoModel.from_pretrained("{repo_name}")
        self.hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(p=dropout_rate)
        self.module_classifier = nn.Linear(self.hidden_size, num_module_labels)
        self.date_classifier = nn.Linear(self.hidden_size, num_date_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)
        module_logits = self.module_classifier(cls_output)
        date_logits = self.date_classifier(cls_output)
        return module_logits, date_logits

# Load model
classifier_config = torch.hub.load_state_dict_from_url(
    f"https://huggingface.co/{repo_name}/resolve/main/classifier_heads.pt"
)

model = TinyBERTDualClassifier(
    num_module_labels=6,
    num_date_labels=7
)

model.module_classifier.load_state_dict(classifier_config['module_classifier'])
model.date_classifier.load_state_dict(classifier_config['date_classifier'])

tokenizer = AutoTokenizer.from_pretrained("{repo_name}")

# Inference
model.eval()
text = "Show my holdings for this month"
inputs = tokenizer(text, return_tensors='pt', padding='max_length',
                   truncation=True, max_length=128)

with torch.no_grad():
    module_logits, date_logits = model(inputs['input_ids'], inputs['attention_mask'])
    module_pred = torch.argmax(module_logits, dim=1).item()
    date_pred = torch.argmax(date_logits, dim=1).item()

module_labels = {module_labels}
date_labels = {date_labels}

print(f"Module: {{module_labels[module_pred]}}")
print(f"Date: {{date_labels[date_pred]}}")
```

## Training Details

- **Dataset**: JayShah07/multi_label_reporting
- **Training Samples**: {len(reporting['train'])}
- **Validation Samples**: {len(reporting['validation'])}
- **Test Samples**: {len(reporting['test'])}
- **Epochs**: {EPOCHS}
- **Batch Size**: {BATCH_SIZE}
- **Learning Rate**: {LEARNING_RATE}
- **Optimizer**: AdamW
- **Loss Function**: CrossEntropyLoss (separate for each head)

## Latency

Average inference latency on sample queries (mean ± std):
- See notebook for detailed latency analysis

## Citation

If you use this model, please cite:
```bibtex
@misc{{tinybert-dual-classifier,
  author = {{Jay Shah}},
  title = {{TinyBERT Dual Classifier for Investment Reporting}},
  year = {{2025}},
  publisher = {{Hugging Face}},
  howpublished = {{\\url{{https://huggingface.co/{repo_name}}}}}
}}
```

## License

Apache 2.0
"""

# Save model card
with open(os.path.join(save_directory, 'README.md'), 'w') as f:
    f.write(model_card)

print("✓ Model card (README.md) created")

In [ ]:
# Upload README.md to Hugging Face
api.upload_file(
    path_or_fileobj=os.path.join(save_directory, 'README.md'),
    path_in_repo='README.md',
    repo_id=repo_name,
    repo_type='model'
)

print(f"✓ Model card uploaded to: https://huggingface.co/{repo_name}")

In [ ]:
# ============================================================================
# OPTIONAL: VISUALIZE TRAINING HISTORY
# ============================================================================

import matplotlib.pyplot as plt

# Extract metrics from training history
epochs_list = [h['epoch'] for h in training_history]
train_losses = [h['train_loss'] for h in training_history]
val_losses = [h['val_loss'] for h in training_history]
module_accs = [h['module_acc'] for h in training_history]
date_accs = [h['date_acc'] for h in training_history]

# Create figure with subplots
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot losses
axes[0].plot(epochs_list, train_losses, marker='o', label='Training Loss')
axes[0].plot(epochs_list, val_losses, marker='s', label='Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

# Plot accuracies
axes[1].plot(epochs_list, module_accs, marker='o', label='Module Accuracy')
axes[1].plot(epochs_list, date_accs, marker='s', label='Date Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Module and Date Classification Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(os.path.join(save_directory, 'training_history.png'), dpi=150)
plt.show()

print(f"✓ Training visualization saved to: {save_directory}/training_history.png")

In [ ]:
# ============================================================================
# SECTION 22: SAVE COMPLETE INFERENCE SCRIPT
# ============================================================================

inference_script = f'''"""
Inference script for TinyBERT Dual Classifier
Usage: python inference.py --text "Show my holdings"
"""

import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from huggingface_hub import hf_hub_download
import argparse

class TinyBERTDualClassifier(nn.Module):
    def __init__(self, num_module_labels, num_date_labels, dropout_rate=0.1):
        super(TinyBERTDualClassifier, self).__init__()
        self.encoder = AutoModel.from_pretrained("{repo_name}")
        self.hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(p=dropout_rate)
        self.module_classifier = nn.Linear(self.hidden_size, num_module_labels)
        self.date_classifier = nn.Linear(self.hidden_size, num_date_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)
        module_logits = self.module_classifier(cls_output)
        date_logits = self.date_classifier(cls_output)
        return module_logits, date_logits

def load_model():
    # Download classifier heads
    classifier_path = hf_hub_download(
        repo_id="{repo_name}",
        filename="classifier_heads.pt"
    )
    classifier_config = torch.load(classifier_path, map_location='cpu')

    # Initialize model
    model = TinyBERTDualClassifier(
        num_module_labels=len(classifier_config['module_labels']),
        num_date_labels=len(classifier_config['date_labels'])
    )

    # Load weights
    model.module_classifier.load_state_dict(classifier_config['module_classifier'])
    model.date_classifier.load_state_dict(classifier_config['date_classifier'])

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained("{repo_name}")

    model.eval()
    return model, tokenizer, classifier_config

def predict(text, model, tokenizer, classifier_config):
    # Tokenize
    inputs = tokenizer(
        text,
        return_tensors='pt',
        padding='max_length',
        truncation=True,
        max_length=128
    )

    # Predict
    with torch.no_grad():
        module_logits, date_logits = model(
            inputs['input_ids'],
            inputs['attention_mask']
        )

        # Get predictions
        module_probs = torch.softmax(module_logits, dim=1)
        date_probs = torch.softmax(date_logits, dim=1)

        module_pred = torch.argmax(module_probs, dim=1).item()
        date_pred = torch.argmax(date_probs, dim=1).item()

        module_conf = module_probs[0, module_pred].item()
        date_conf = date_probs[0, date_pred].item()

    return {{
        'module': classifier_config['module_labels'][module_pred],
        'module_confidence': module_conf,
        'date': classifier_config['date_labels'][date_pred],
        'date_confidence': date_conf
    }}

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument('--text', type=str, required=True, help='Input text query')
    args = parser.parse_args()

    print("Loading model...")
    model, tokenizer, config = load_model()

    print(f"\\nInput: {{args.text}}")
    result = predict(args.text, model, tokenizer, config)

    print(f"\\nPredictions:")
    print(f"  Module: {{result['module']}} (confidence: {{result['module_confidence']:.4f}})")
    print(f"  Date: {{result['date']}} (confidence: {{result['date_confidence']:.4f}})")
'''

# Save inference script
with open(os.path.join(save_directory, 'inference.py'), 'w') as f:
    f.write(inference_script)

print("✓ Inference script saved to: ./tinybert_dual_classifier/inference.py")
print("\nYou can use it with:")
print('  python inference.py --text "Show my holdings for this month"')

In [ ]:
# ============================================================================
# OPTIONAL: CREATE CONFUSION MATRICES
# ============================================================================

from sklearn.metrics import confusion_matrix
import seaborn as sns

# Create confusion matrices for test set
module_cm = confusion_matrix(test_metrics['module_labels'], test_metrics['module_preds'])
date_cm = confusion_matrix(test_metrics['date_labels'], test_metrics['date_preds'])

# Plot confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Module confusion matrix
sns.heatmap(module_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=module_labels, yticklabels=module_labels,
            ax=axes[0])
axes[0].set_title('Module Classification Confusion Matrix')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')
plt.setp(axes[0].get_xticklabels(), rotation=45, ha='right')
plt.setp(axes[0].get_yticklabels(), rotation=0)

# Date confusion matrix
sns.heatmap(date_cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=date_labels, yticklabels=date_labels,
            ax=axes[1])
axes[1].set_title('Date Classification Confusion Matrix')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')
plt.setp(axes[1].get_xticklabels(), rotation=45, ha='right')
plt.setp(axes[1].get_yticklabels(), rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(save_directory, 'confusion_matrices.png'), dpi=150)
plt.show()

print(f"✓ Confusion matrices saved to: {save_directory}/confusion_matrices.png")

In [ ]:
# ============================================================================
# COMPREHENSIVE PYTORCH PARAMETERS EXPLANATION
# ============================================================================

print("="*80)
print("PYTORCH PARAMETERS & CONCEPTS - DETAILED EXPLANATION")
print("="*80)

explanation = """

╔══════════════════════════════════════════════════════════════════════════════╗
║                    1. MODEL ARCHITECTURE PARAMETERS                          ║
╚══════════════════════════════════════════════════════════════════════════════╝

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔹 nn.Module
   - Base class for ALL neural networks in PyTorch
   - Provides: forward(), backward(), parameters(), state_dict()
   - Usage: class MyModel(nn.Module)
   - Why: Automatic gradient computation, parameter management, GPU support

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔹 nn.Linear(in_features, out_features, bias=True)
   - Fully connected layer: y = xW^T + b
   - in_features: Input dimension (e.g., 312 for TinyBERT hidden size)
   - out_features: Output dimension (e.g., 5 for module classes)
   - bias: Whether to add bias term (default: True)
   - Parameters: Weight matrix [out_features, in_features] + bias [out_features]

   Example:
   linear = nn.Linear(312, 5)  # 312*5 + 5 = 1,565 parameters
   x = torch.randn(16, 312)    # Batch of 16, dimension 312
   y = linear(x)               # Output: [16, 5]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔹 nn.Dropout(p=0.1, inplace=False)
   - Regularization: Randomly zeros elements during training
   - p: Probability of dropping (0.0 to 1.0). p=0.1 means 10% dropout
   - inplace: Modify input tensor directly (saves memory)
   - Behavior:
     * Training: Randomly set p% of elements to 0, scale rest by 1/(1-p)
     * Evaluation: Pass through unchanged (no dropout)

   Why dropout?
   - Prevents overfitting by forcing network to learn redundant representations
   - Acts like ensemble of many networks
   - Only active during training (disabled in eval mode)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔹 hidden_size / embedding_dim
   - Dimension of internal representations
   - TinyBERT: 312, BERT-base: 768, BERT-large: 1024
   - Larger = more capacity but slower and more memory
   - Trade-off: Model capacity vs computational cost

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


╔══════════════════════════════════════════════════════════════════════════════╗
║                    2. TOKENIZATION PARAMETERS                                ║
╚══════════════════════════════════════════════════════════════════════════════╝

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔹 padding (str or bool)
   - 'max_length': Pad all sequences to max_length
   - 'longest': Pad to longest sequence in batch
   - True: Same as 'longest'
   - False: No padding

   Why: Neural networks need fixed-size inputs for batching

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔹 truncation (bool)
   - True: Cut sequences longer than max_length
   - False: Raise error if sequence too long

   Why: BERT models have max input length (usually 512 tokens)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔹 max_length (int)
   - Maximum sequence length in tokens
   - Our notebook: 128 tokens
   - Trade-off:
     * Shorter (64-128): Faster, less memory, may lose context
     * Longer (256-512): Slower, more memory, preserves context

   Rule of thumb:
   - Short texts (queries): 64-128
   - Medium texts (paragraphs): 256-384
   - Long texts (documents): 512+

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔹 return_tensors (str or None)
   - 'pt': Return PyTorch tensors
   - 'tf': Return TensorFlow tensors
   - 'np': Return NumPy arrays
   - None: Return Python lists

   Use 'pt' when working with PyTorch models

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔹 attention_mask
   - Binary tensor: 1 for real tokens, 0 for padding
   - Shape: [batch_size, seq_length]
   - Purpose: Tell model which tokens to attend to

   Example:
   Input: "Show holdings [PAD] [PAD]"
   Mask:  [1, 1, 0, 0]  → Model ignores padding

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


╔══════════════════════════════════════════════════════════════════════════════╗
║                    3. TRAINING HYPERPARAMETERS                               ║
╚══════════════════════════════════════════════════════════════════════════════╝

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔹 batch_size (int)
   - Number of samples processed together
   - Our notebook: 16

   Effects:
   ┌─────────────┬──────────────┬─────────────┬──────────────┐
   │ Batch Size  │ Memory       │ Speed       │ Convergence  │
   ├─────────────┼──────────────┼─────────────┼──────────────┤
   │ Small (8)   │ Low          │ Slow        │ Noisy/Good   │
   │ Medium (16) │ Medium       │ Medium      │ Balanced     │
   │ Large (32+) │ High         │ Fast        │ Smooth/Poor  │
   └─────────────┴──────────────┴─────────────┴──────────────┘

   How to choose:
   - Start with 16 or 32
   - Increase if you have GPU memory
   - Decrease if getting Out of Memory (OOM) errors
   - Powers of 2 are optimal (8, 16, 32, 64)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔹 epochs (int)
   - Number of complete passes through training data
   - Our notebook: 10

   Guidelines:
   - Small dataset (<1000): 20-50 epochs
   - Medium dataset (1000-10000): 10-20 epochs
   - Large dataset (10000+): 3-10 epochs

   Watch for:
   - Underfitting: Training loss still decreasing → More epochs
   - Overfitting: Val loss increasing, train loss decreasing → Fewer epochs

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔹 learning_rate (float)
   - Step size for parameter updates
   - Our notebook: 2e-5 (0.00002)

   Typical ranges:
   - Training from scratch: 1e-3 to 1e-4
   - Fine-tuning pre-trained: 1e-5 to 5e-5
   - Very large models: 1e-6 to 1e-5

   Effects:
   ┌────────────┬──────────────────────────────────────┐
   │ Too High   │ Loss explodes, NaN values, unstable  │
   │ Too Low    │ Very slow training, may get stuck    │
   │ Just Right │ Smooth decrease in loss              │
   └────────────┴──────────────────────────────────────┘

   Formula: new_param = old_param - learning_rate * gradient

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔹 weight_decay (float)
   - L2 regularization coefficient
   - Our notebook: 0.01

   Purpose: Prevents overfitting by penalizing large weights
   Formula: loss = original_loss + (weight_decay / 2) * sum(weight²)

   Typical values:
   - No regularization: 0.0
   - Light regularization: 0.001 to 0.01
   - Strong regularization: 0.01 to 0.1

   When to use:
   - Large weight_decay: Small dataset, complex model (high overfitting risk)
   - Small weight_decay: Large dataset, simple model

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


╔══════════════════════════════════════════════════════════════════════════════╗
║                    4. OPTIMIZER PARAMETERS (AdamW)                           ║
╚══════════════════════════════════════════════════════════════════════════════╝

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔹 torch.optim.AdamW(params, lr, betas, eps, weight_decay)

   AdamW = Adam with decoupled Weight decay
   - Better for transformer models than standard Adam

   Parameters:

   • lr (float): Learning rate (default: 1e-3)
     - Main control for step size

   • betas (tuple): (beta1, beta2) (default: (0.9, 0.999))
     - beta1: Exponential decay for 1st moment (mean of gradients)
     - beta2: Exponential decay for 2nd moment (variance of gradients)
     - Higher beta1 → More momentum, smoother updates
     - Higher beta2 → More stable, less sensitive to recent gradients

   • eps (float): Numerical stability constant (default: 1e-8)
     - Prevents division by zero
     - Rarely needs changing

   • weight_decay (float): L2 penalty coefficient (default: 0.01)
     - Decoupled from gradient in AdamW (better than Adam)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔹 optimizer.zero_grad()
   - Resets gradients to zero
   - MUST be called before each backward pass

   Why: PyTorch accumulates gradients by default
   Without zero_grad(): Gradients from previous batches would add up

   Flow:
   1. optimizer.zero_grad()  ← Clear old gradients
   2. loss.backward()        ← Compute new gradients
   3. optimizer.step()       ← Update parameters

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔹 optimizer.step()
   - Updates model parameters using computed gradients

   What it does:
   For each parameter p with gradient g:
   p = p - learning_rate * g  (simplified)

   Actual AdamW update is more complex:
   - Computes adaptive learning rates per parameter
   - Uses momentum (running average of gradients)
   - Applies weight decay

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


╔══════════════════════════════════════════════════════════════════════════════╗
║                    5. LEARNING RATE SCHEDULER                                ║
╚══════════════════════════════════════════════════════════════════════════════╝

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔹 get_linear_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps)

   Two phases:

   1. WARMUP (first num_warmup_steps):
      - Linearly increase LR from 0 to initial_lr
      - Prevents large updates early in training
      - Typical: 5-10% of total steps

   2. LINEAR DECAY (remaining steps):
      - Linearly decrease LR from initial_lr to 0
      - Helps fine-tune in later stages

   Visualization:

   LR │
      │    ╱╲
      │   ╱  ╲
      │  ╱    ╲___
      │ ╱         ╲___
      │╱              ╲___
      └────────────────────────→ Steps
        ↑              ↑
      warmup        training

   Why it helps:
   - Warmup: Prevents exploding gradients early
   - Decay: Better convergence to local minima

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔹 num_training_steps
   - Total number of optimizer updates
   - Calculation: len(train_dataloader) * num_epochs
   - Example: 16 batches * 10 epochs = 160 steps

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔹 num_warmup_steps
   - Number of warmup steps (linear increase)
   - Typical: 5-10% of num_training_steps
   - Example: 0.1 * 160 = 16 warmup steps

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔹 scheduler.step()
   - Updates learning rate according to schedule
   - MUST be called after optimizer.step()

   Flow per training step:
   1. optimizer.zero_grad()
   2. loss.backward()
   3. optimizer.step()      ← Update parameters
   4. scheduler.step()      ← Update learning rate

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


╔══════════════════════════════════════════════════════════════════════════════╗
║                    6. LOSS FUNCTION PARAMETERS                               ║
╚══════════════════════════════════════════════════════════════════════════════╝

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔹 nn.CrossEntropyLoss(weight, ignore_index, reduction)

   Combines LogSoftmax + NLLLoss in one operation

   Formula:
   loss = -log(softmax(logits)[true_class])
        = -log(exp(logits[true_class]) / sum(exp(logits)))

   Parameters:

   • weight (Tensor or None): Class weights for imbalanced datasets
     - Shape: [num_classes]
     - Example: [0.5, 1.0, 2.0] → Class 2 is weighted more

   • ignore_index (int): Ignore specific class (default: -100)
     - Useful for padding or unknown classes

   • reduction (str): How to combine losses (default: 'mean')
     - 'mean': Average loss over batch
     - 'sum': Sum all losses
     - 'none': Return loss per sample

   Input format:
   - Logits: Raw scores [batch_size, num_classes]
   - Targets: Class indices [batch_size] (NOT one-hot!)

   Example:
   logits = [[2.0, 1.0, 0.1]]  # 3 classes
   target = [0]                 # True class is 0
   loss = -log(exp(2.0) / (exp(2.0) + exp(1.0) + exp(0.1)))

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


╔══════════════════════════════════════════════════════════════════════════════╗
║                    7. GRADIENT COMPUTATION                                   ║
╚══════════════════════════════════════════════════════════════════════════════╝

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔹 loss.backward()
   - Computes gradients via backpropagation
   - Gradient: How much loss changes if parameter changes

   What happens:
   1. PyTorch builds computation graph during forward pass
   2. backward() traverses graph in reverse
   3. Applies chain rule to compute gradients
   4. Stores gradients in parameter.grad

   Example:
   loss = (prediction - target)²
   gradient = 2 * (prediction - target) * d(prediction)/d(weight)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔹 torch.nn.utils.clip_grad_norm_(parameters, max_norm, norm_type=2)

   Prevents exploding gradients by scaling them down

   Parameters:
   • max_norm (float): Maximum allowed gradient norm (typical: 0.5 to 5.0)
   • norm_type (float): Type of norm (2 = L2 norm, default)

   What it does:
   1. Compute total gradient norm: ||g|| = sqrt(sum(g_i²))
   2. If ||g|| > max_norm:
      Scale all gradients: g = g * (max_norm / ||g||)

   Why needed:
   - Deep networks can have exploding gradients
   - Especially in RNNs and transformers
   - Clipping prevents NaN/Inf values

   Visual:
   Before clipping:  →→→→→→→→→→→→→ (gradient too large)
   After clipping:   →→→→ (scaled to max_norm)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


╔══════════════════════════════════════════════════════════════════════════════╗
║                    8. MODEL MODES & GRADIENT CONTROL                         ║
╚══════════════════════════════════════════════════════════════════════════════╝

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔹 model.train()
   - Sets model to training mode

   Effects:
   • Enables Dropout: Randomly drops neurons
   • Enables BatchNorm training: Updates running statistics
   • Affects other layers with different train/eval behavior

   When to use: Before training loop

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔹 model.eval()
   - Sets model to evaluation mode

   Effects:
   • Disables Dropout: Uses all neurons
   • Disables BatchNorm training: Uses fixed statistics
   • More deterministic predictions

   When to use: Before validation/testing/inference

   CRITICAL: Always use eval() for evaluation!
   Without eval(): Different results each run due to dropout

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔹 torch.no_grad()
   - Context manager that disables gradient computation

   Benefits:
   • Reduces memory usage (no computation graph)
   • Speeds up inference (no gradient tracking)
   • Prevents accidental parameter updates

   Usage:
   with torch.no_grad():
       predictions = model(inputs)  # No gradients computed

   When to use:
   - Validation/testing
   - Inference
   - Any time you don't need gradients

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔹 parameter.requires_grad
   - Boolean flag: Whether to compute gradients

   True (default): Parameter will be updated during training
   False: Parameter is frozen (no gradient computation)

   Usage - Freeze layers:
   for param in model.encoder.parameters():
       param.requires_grad = False  # Freeze encoder

   Why freeze:
   - Fine-tune only classification head
   - Faster training
   - Prevent catastrophic forgetting

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


╔══════════════════════════════════════════════════════════════════════════════╗
║                    9. TENSOR OPERATIONS                                      ║
╚══════════════════════════════════════════════════════════════════════════════╝

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔹 torch.argmax(tensor, dim)
   - Returns index of maximum value

   Parameters:
   • dim (int): Dimension to reduce
     - dim=0: Max across rows (column-wise)
     - dim=1: Max across columns (row-wise)
     - dim=-1: Last dimension

   Example:
   logits = [[2.1, 0.5, 1.2],    # Batch of 2, 3 classes
             [0.3, 1.8, 0.9]]

   torch.argmax(logits, dim=1) = [0, 1]  # Class 0 for sample 1, class 1 for sample 2

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔹 torch.softmax(tensor, dim)
   - Converts logits to probabilities

   Formula: softmax(x_i) = exp(x_i) / sum(exp(x_j))

   Properties:
   • All outputs are between 0 and 1
   • Sum of outputs = 1.0 (valid probability distribution)

   Example:
   logits = [2.0, 1.0, 0.1]
   probs = softmax(logits) = [0.659, 0.242, 0.099]  # Sum = 1.0

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔹 tensor.item()
   - Converts single-element tensor to Python scalar

   Usage:
   loss_tensor = torch.tensor([3.14])
   loss_value = loss_tensor.item()  # 3.14 (float)

   Only works for tensors with one element!

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔹 tensor.cpu() / tensor.cuda()
   - Move tensor between devices

   cpu(): GPU → CPU
   cuda(): CPU → GPU (if available)
   to(device): Move to specified device

   Example:
   x_gpu = torch.tensor([1, 2, 3]).cuda()
   x_cpu = x_gpu.cpu()

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔹 tensor.numpy()
   - Converts PyTorch tensor to NumPy array

   Note: Tensor must be on CPU first!
   x_gpu.cpu().numpy()  ✓ Correct
   x_gpu.numpy()